In [1]:
# @title Connect google drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# @title STEP:1 set env TODAY_WORK_IN as a today date with year
from datetime import datetime

TODAY_WORK_IN = datetime.now().strftime("%d%b%Y").upper()

# MUST run in notebook
%env TODAY_WORK_IN={TODAY_WORK_IN}

print("Set:", TODAY_WORK_IN)

In [ ]:
# @title STEP:2 get TODAY_WORK_IN and create dir
%%bash
echo $TODAY_WORK_IN

DIR="/content/$TODAY_WORK_IN"

if [ -d "$DIR" ]; then
  echo "📁 Exists: $DIR"
else
  mkdir -p "$DIR"
  echo "✅ Created: $DIR"
fi

cd "$DIR"
pwd
env | grep TODAY_WORK_IN

In [ ]:
# @title STEP:3 get all env variables

# !env
# grep multiple variable in single with |
!printenv | grep -E "HF_|CUDA|TODAY_WORK_IN"
%cd /content/$TODAY_WORK_IN
!pwd

In [ ]:
import os

# print(os.environ)
print("TODAY_WORK_IN:", os.getenv("TODAY_WORK_IN"))
print("HF_HOME:", os.getenv("HF_HOME"))
print("CUDA_VISIBLE_DEVICES:", os.getenv("CUDA_VISIBLE_DEVICES"))

In [ ]:
# @title Install system dependencies
!pwd
!apt update -y
!apt install -y build-essential git wget
!pip install -q cmake

In [ ]:
# @title Verify GPU (must work)
!nvidia-smi

In [ ]:
# @title Clone fresh llama.cpp
# %cd /content
!pwd
%cd /content/$TODAY_WORK_IN
# !pwd
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

In [ ]:
# @title Build with GPU (MOST STABLE METHOD)

!cmake -B build \
  -DGGML_CUDA=ON \
  -DCMAKE_CUDA_ARCHITECTURES=native

In [ ]:
# @title Build with CPU
# !pwd
%cd /content/$TODAY_WORK_IN/llama.cpp
!cmake -B build
# !pwd
# %cd /content/llama.cpp

In [ ]:
# @title same for both CPU and GPU
# %cd /content/llama.cpp
%cd /content/$TODAY_WORK_IN/llama.cpp
!cmake --build build -j$(nproc)

In [ ]:
# @title Test binary (important sanity check)
%cd /content/$TODAY_WORK_IN/llama.cpp
!./build/bin/llama-cli -h

In [ ]:
# @title (Optional but recommended) Login to Hugging Face
# !pip install -q huggingface_hub
# !huggingface-cli login

In [ ]:
# @title copy hf-cache from drive if already downloaded
# %cd /content/$TODAY_WORK_IN
!cp -r /content/drive/MyDrive/hf-cache /content/$TODAY_WORK_IN

# then use
%env HF_HOME=/content/$TODAY_WORK_IN/hf-cache

In [ ]:
# @title Set huggingface home env var
# %env HF_HOME=/content/drive/MyDrive/hf-cache
# %env HF_HOME=/content/hf-cache
# %env HF_HOME=/content/$TODAY_WORK_IN/hf-cache

In [ ]:
# @title download model from  Hugging Face into HF_HOME location
# !./build/bin/llama-cli -hf ggml-org/gemma-4-E2B-it-GGUF -n 10

In [ ]:
# @title copy model folder local to drive
# !cp -r /content/hf-cache /content/drive/MyDrive

# then use
# %env HF_HOME=/content/hf-cache

In [ ]:
# @title Start llama-server (GPU + logs)
%cd /content/$TODAY_WORK_IN/llama.cpp
!pwd
# run on 8081 port
!./build/bin/llama-server \
  -hf ggml-org/gemma-4-E2B-it-GGUF \
  -ngl 99 \
  --flash-attn on \
  --host 0.0.0.0 \
  --port 8081 \
  > llama.log 2>&1 &

In [ ]:
# @title Verify server is running
!curl http://localhost:8081/v1/models | jq

# curl 'http://127.0.0.1:8081/v1/chat/completions' \
#   -H 'Accept: */*' \
#   -H 'Accept-Language: en-GB,en-US;q=0.9,en;q=0.8' \
#   -H 'Connection: keep-alive' \
#   -H 'Content-Type: application/json' \
#   --data-raw $'{"messages":[{"role":"user","content":"explain about vactor database?"}],"stream":true,"return_progress":true,"reasoning_format":"auto","timings_per_token":true}'

In [ ]:
# @title Download cloudflared (for public URL)
%cd /content/$TODAY_WORK_IN
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
# @title Start public tunnel
%cd /content/$TODAY_WORK_IN
!./cloudflared-linux-amd64 tunnel --url http://localhost:8081
# !./cloudflared-linux-amd64 tunnel --url http://localhost:8081 > cloudflared.log 2>&1 &

Ollama setup

In [ ]:
# @title find gguf models
# %cd /content/$TODAY_WORK_IN
!find /content/$TODAY_WORK_IN/hf-cache -name "*.gguf"
# !find /content/drive/MyDrive/hf-cache -name "*.gguf"

In [ ]:

# @title Copy model to clean folder ref only
!mkdir -p ~/ollama-models/gemma
!cp /content/hf-cache/hub/models--ggml-org--gemma-4-E2B-it-GGUF/snapshots/*/gemma-4-E2B-it-Q8_0.gguf ~/ollama-models/gemma/
!cd ~/ollama-models/gemma

In [ ]:

# @title Copy model to clean folder
# !mkdir -p ~/ollama-models/gemma
# !pwd
!mkdir -p /content/$TODAY_WORK_IN/ollama-models/gemma
!cp /content/$TODAY_WORK_IN/hf-cache/hub/models--ggml-org--gemma-4-E2B-it-GGUF/snapshots/*/gemma-4-E2B-it-Q8_0.gguf /content/$TODAY_WORK_IN/ollama-models/gemma
!cd /content/$TODAY_WORK_IN/ollama-models/gemma
# !cd ~/ollama-models/gemma

# run in terminal
## go to dir
cd /content/ollama-models/gemma
## add model name
MODEL="gemma-4-E2B-it-Q8_0.gguf"
## create Modelfile file
cat << EOF > Modelfile
FROM ./$MODEL

PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20

SYSTEM "You are a helpful AI assistant."
EOF

In [ ]:
# @title list of all gguf model from '/content/hf-cache' and create ollama model into '/content/ollama_models' OLD
import os
import shutil
from pathlib import Path

# ===== CONFIG =====
SOURCE_DIR = Path("/content/hf-cache")        # where GGUF exists
OUTPUT_BASE = Path("/content/ollama_models")  # output folder

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# ===== FIND ALL GGUF FILES =====
gguf_files = list(SOURCE_DIR.rglob("*.gguf"))

# Filter only main models (exclude mmproj)
main_models = [f for f in gguf_files if "mmproj" not in f.name]

if not main_models:
    raise Exception("❌ No valid GGUF models found!")

print(f"🔍 Found {len(main_models)} models")

# ===== GENERATE MODELS =====
for model_path in main_models:
    model_name = model_path.stem   # filename without extension
    model_dir = OUTPUT_BASE / model_name

    model_dir.mkdir(parents=True, exist_ok=True)

    # Copy model
    target_model = model_dir / model_path.name
    shutil.copy2(model_path, target_model)

    # Create Modelfile
    modelfile_path = model_dir / "Modelfile"

    content = f"""FROM ./{target_model.name}

PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20

SYSTEM "You are a helpful AI assistant."
"""

    with open(modelfile_path, "w") as f:
        f.write(content)

    print(f"✅ Created: {model_name}")

print("\n🎉 All models ready!")
print(f"📁 Output: {OUTPUT_BASE}")

In [ ]:
# @title list of all gguf model from '/content/hf-cache' and create ollama model into '/content/ollama_models' NEW
import os
import shutil
from pathlib import Path
import re

# ===== CONFIG =====
# SOURCE_DIR = Path("/content/$TODAY_WORK_IN/hf-cache")
# OUTPUT_BASE = Path("/content/$TODAY_WORK_IN/ollama_models")


SOURCE_DIR = Path(f"/content/{TODAY_WORK_IN}/hf-cache")
OUTPUT_BASE = Path(f"/content/{TODAY_WORK_IN}/ollama_models")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# ===== HELPERS =====

def extract_quant_info(filename):
    """
    Extract quantization info like Q4_K_M, Q8_0, etc.
    """
    match = re.search(r'(Q\d(?:_[A-Z]+)?)', filename)
    return match.group(1) if match else "UNKNOWN"


def get_generation_params(quant):
    """
    Smart parameter tuning based on quantization
    """
    # Default values
    params = {
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 40
    }

    if "Q2" in quant:
        params.update({"temperature": 0.8, "top_p": 0.9, "top_k": 60})
    elif "Q3" in quant:
        params.update({"temperature": 0.75, "top_p": 0.92, "top_k": 50})
    elif "Q4" in quant:
        params.update({"temperature": 0.7, "top_p": 0.93, "top_k": 40})
    elif "Q5" in quant:
        params.update({"temperature": 0.65, "top_p": 0.94, "top_k": 30})
    elif "Q6" in quant:
        params.update({"temperature": 0.6, "top_p": 0.95, "top_k": 25})
    elif "Q8" in quant:
        params.update({"temperature": 0.5, "top_p": 0.97, "top_k": 20})

    return params


def normalize_model_name(name):
    """
    Clean model name for Ollama
    """
    name = name.lower()
    name = re.sub(r'[^a-z0-9\-\.]', '-', name)
    name = re.sub(r'-+', '-', name)
    return name.strip("-")


# ===== FIND MODELS =====
gguf_files = list(SOURCE_DIR.rglob("*.gguf"))

# Filter only main models
main_models = [f for f in gguf_files if "mmproj" not in f.name]

if not main_models:
    raise Exception("❌ No GGUF models found!")

print(f"🔍 Found {len(main_models)} GGUF files")

# ===== REMOVE DUPLICATES (keep best quant per base model) =====
model_map = {}

for model in main_models:
    base_name = re.sub(r'-Q\d.*', '', model.stem)
    quant = extract_quant_info(model.name)

    # Prefer higher quant (Q8 > Q6 > Q4)
    score = int(re.search(r'Q(\d)', quant).group(1)) if "Q" in quant else 0

    if base_name not in model_map or model_map[base_name]["score"] < score:
        model_map[base_name] = {
            "path": model,
            "quant": quant,
            "score": score
        }

selected_models = list(model_map.values())

print(f"🎯 Selected {len(selected_models)} best models")

# ===== GENERATE =====
for item in selected_models:
    model_path = item["path"]
    quant = item["quant"]

    base_name = re.sub(r'-Q\d.*', '', model_path.stem)
    clean_name = normalize_model_name(base_name)

    model_dir = OUTPUT_BASE / clean_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # Copy model
    target_model = model_dir / model_path.name
    shutil.copy2(model_path, target_model)

    # Get smart params
    params = get_generation_params(quant)

    # Create Modelfile
    modelfile_path = model_dir / "Modelfile"

    content = f"""FROM ./{target_model.name}

PARAMETER temperature {params['temperature']}
PARAMETER top_p {params['top_p']}
PARAMETER top_k {params['top_k']}

SYSTEM "You are a highly intelligent and helpful AI assistant."
"""

    with open(modelfile_path, "w") as f:
        f.write(content)

    print(f"✅ Created: {clean_name} ({quant})")

print("\n🎉 ALL MODELS READY!")
print(f"📁 Output: {OUTPUT_BASE}")

In [ ]:
# @title get all models with there name and size
%cd /content/$TODAY_WORK_IN/ollama_models
!du -sh *

In [ ]:
# @title install ollama
# update
!apt-get update
# install required packeg
!apt-get install -y zstd
# install ollama
!curl -fsSL https://ollama.com/install.sh | sh
# check version
!ollama --version

In [ ]:
# @title ollama serve
# !ollama serve &
!ollama serve > ollama.log 2>&1 &

In [ ]:
!ollama list

In [ ]:
# @title go to model dir and make a new model from into ollama

model_dir = "gemma-4-e2b-it"
new_model_name = "gemma-sandeep"

!ollama create {new_model_name} -f /content/$TODAY_WORK_IN/ollama_models/{model_dir}/Modelfile
!ollama list

In [ ]:
# @title run
!ollama run gemma-sandeep:latest
# !ollama list
# !ollama rm gemma-local

In [ ]:
!ollama rm gemma-local:latest

In [ ]:
!pwd
%cd /content/$TODAY_WORK_IN
!pwd

In [ ]:
# @title create colab_download.py to zip any folder for download or backup
%%writefile colab_download.py
#!/usr/bin/env python3

import argparse
import shutil
from pathlib import Path
import sys

# ===== Detect real Colab notebook =====
def is_colab_notebook():
    try:
        import IPython
        from google.colab import files  # noqa

        ip = IPython.get_ipython()
        if ip is None:
            return False

        # Must have kernel + comm_manager (UI available)
        if not hasattr(ip, "kernel") or ip.kernel is None:
            return False

        return True
    except Exception:
        return False


IN_COLAB_NOTEBOOK = is_colab_notebook()


# ===== Zip function =====
def zip_folder(input_path: Path, output_name: str):
    if not input_path.exists():
        print(f"❌ Path not found: {input_path}")
        sys.exit(1)

    zip_path = shutil.make_archive(output_name, "zip", input_path)
    print(f"📦 Zip created: {zip_path}")
    return zip_path


# ===== Download function =====
def download_file(zip_file: str):
    if not IN_COLAB_NOTEBOOK:
        print("⚠️ Not running inside Colab notebook UI")
        print(f"📁 File saved locally: {zip_file}")
        return

    try:
        from google.colab import files
        print("⬇️ Downloading via Colab UI...")
        files.download(zip_file)
    except Exception as e:
        print("❌ Download failed:", str(e))
        print(f"📁 You can manually access file: {zip_file}")


# ===== Main =====
def main():
    parser = argparse.ArgumentParser(
        description="Zip and optionally download a folder (Colab + Local compatible)"
    )

    parser.add_argument(
        "--input",
        required=True,
        help="Folder to zip (e.g. /content/ollama_models)"
    )

    parser.add_argument(
        "--output",
        default="output",
        help="Output zip name (without .zip)"
    )

    parser.add_argument(
        "--download",
        action="store_true",
        help="Download (only works in Colab notebook)"
    )

    args = parser.parse_args()

    input_path = Path(args.input).resolve()
    output_name = args.output

    zip_file = zip_folder(input_path, output_name)

    # ===== Download only if explicitly requested =====
    if args.download:
        download_file(zip_file)
    else:
        print("ℹ️ Download skipped (use --download to enable)")


if __name__ == "__main__":
    main()

In [ ]:
# @title It will create zip folder
%cd /content/$TODAY_WORK_IN
!python colab_download.py \
  --input /content/$TODAY_WORK_IN/ollama_models \
  --output ollama_models \
  --download

In [ ]:
# @title manual download zip
from google.colab import files
SOURCE_DIR = Path(f"/content/{TODAY_WORK_IN}/ollama_models.zip")
files.download(SOURCE_DIR)
# files.download("/content/ollama_models.zip")